Visualisation for research question 1

Relevant imports:

In [5]:
import pandas as pd
import numpy as np
import plotly.express as px

In [6]:
df = pd.read_csv("hamburg_birdCounts_pollution_2021-2025.csv")

Bar Chart Visualisation:

In [7]:
# Prepare date column
df["Date"] = pd.to_datetime(df["Date"])


# Settings
pollutant = "PM2.5_mean"
# Critical Value
threshold = 25
n_bins = 8


# Keep only required columns
plot_df = df[[pollutant, "Total_Amount"]].dropna()


# Create bin edges
bin_edges = np.linspace(
    plot_df[pollutant].min(),
    plot_df[pollutant].max(),
    9,
)


# §LLM Help : To assign pollution intervals
plot_df["pollution_bin"] = pd.cut(
    plot_df[pollutant],
    bins=bin_edges,
    include_lowest=True,
)


# §LLM Help: for Aggregating values per interval
summary = (
    plot_df.groupby("pollution_bin", observed=True)
    .agg(
        avg_birds=("Total_Amount", "mean"),
        median_birds=("Total_Amount", "median"),
        days=("Total_Amount", "size"),
        avg_pollution=(pollutant, "mean"),
    )
    .reset_index()
)


# §LLM Help: Extract interval boundaries
bin_left = np.array([interval.left for interval in summary["pollution_bin"]])
bin_right = np.array([interval.right for interval in summary["pollution_bin"]])


# Compute bar position and width
bin_mid = (bin_left + bin_right) / 2
bin_width = bin_right - bin_left


# Color logic
colors = np.where(
    bin_mid >= threshold,
    "#ff9ecf",
    "#a7c7ff",
)


# Interval labels
bin_labels = (
    pd.Series(bin_left).round(1).astype(str)
    + "–"
    + pd.Series(bin_right).round(1).astype(str)
)


# Text values
avg_birds_text = summary["avg_birds"].round(1)
median_birds_text = summary["median_birds"].round(1)
avg_pollution_text = summary["avg_pollution"].round(2)

bar_text = (
    summary["avg_birds"].round(1).astype(str)
    + "<br>"
    + summary["days"].astype(str)
    + " days"
)


# Build Plotly bar
fig = px.bar(
    summary,
    x=bin_mid,
    y="avg_birds",
    text=bar_text,
    title="Average Bird Observations by PM2.5 Level in Hamburg",
    labels={
        "x": "PM2.5 mean (µg/m³)",
        "avg_birds": "Average bird observations",
    },
)


# Bar styling
fig.update_traces(
    marker_color=colors,
    width=bin_width * 0.9,
    textposition="outside",
    # §LLM Help for Hover Label
    hovertemplate=(
        "<b>PM2.5 interval</b>: %{customdata[0]}"
        "<br><b>Average bird observations</b>: %{customdata[1]}"
        "<br><b>Median bird observations</b>: %{customdata[2]}"
        "<br><b>Number of days</b>: %{customdata[3]}"
        "<br><b>Average PM2.5 in interval</b>: %{customdata[4]}"
        "<extra></extra>"
    ),
    customdata=np.column_stack(
        [
            bin_labels,
            avg_birds_text,
            median_birds_text,
            summary["days"],
            avg_pollution_text,
        ]
    ),
)


# Threshold line
fig.add_vline(
    x=threshold,
    line_dash="dash",
    line_width=2,
    line_color="#ff69b4",
    annotation_text=f"Critical value ({threshold} µg/m³)",
    annotation_position="top right",
)


# Highlight critical range
fig.add_vrect(
    x0=threshold,
    x1=bin_right.max(),
    fillcolor="#ff9ecf",
    opacity=0.08,
    line_width=0,
)


# Layout
fig.update_layout(
    template="plotly_white",
    bargap=0.15,
    height=600,
    showlegend=False,
)


# Replace numeric ticks with interval labels
fig.update_xaxes(
    tickmode="array",
    tickvals=bin_mid,
    ticktext=bin_labels,
)


fig.show()